# Hallucination Verifier — Recall & Precision

PDF Module F + §6.1: "summaries contain no hallucinated facts". This notebook injects 10 known-hallucination summaries and 10 grounded summaries against the same listing and reports the verifier's recall / precision.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
os.environ.setdefault('CACHE_DIR', str(Path.cwd() / '.eval_cache'))
from app.models.business import BusinessListing, ReviewData
from app.modules.summarizer import _verify_grounding

listing = BusinessListing(
    id='t', name="Joe's Pizza", address='123 Main St, Brooklyn, NY',
    category='restaurant', source='overpass',
    review_data=ReviewData(
        total_reviews=42, average_rating=4.5, positive_percentage=88.0,
        recurring_themes=['great food','good value'],
        sample_reviews=['best pizza in Brooklyn','thin crust is amazing'],
    ),
)

grounded = [
    "Joe's Pizza on Main St is rated 4.5 with 42 reviews.",
    "Joe's Pizza is praised for great food and good value.",
    "Located in Brooklyn, Joe's Pizza has 42 reviews.",
    "Joe's Pizza serves thin crust pizza in Brooklyn.",
    "Joe's Pizza scores 4.5 from 42 reviewers, highlighting great food.",
    "Joe's Pizza is a restaurant at 123 Main St.",
    "Joe's Pizza in Brooklyn earns strong reviews for thin crust.",
    "Patrons of Joe's Pizza commend the great food and good value.",
    "Joe's Pizza has a rating of 4.5 out of 5.",
    "Joe's Pizza is loved for its amazing thin crust.",
]

hallucinated = [
    "Joe's Pizza won a Michelin Star in 2020.",
    "Joe's Pizza is owned by Gordon Ramsay.",
    "Joe's Pizza serves 99 different toppings.",
    "Joe's Pizza was featured on Food Network.",
    "Joe's Pizza is rated 5.0 by 1000 customers.",
    "Joe's Pizza is in Manhattan, not Brooklyn.",
    "Joe's Pizza opened in 1965 by Italian immigrants.",
    "Joe's Pizza is the best pizza in Chicago.",
    "Joe's Pizza was awarded the James Beard Prize.",
    "Joe's Pizza appears in the Netflix show Chef's Table.",
]

tp = sum(1 for s in hallucinated if not _verify_grounding(s, listing))
fn = len(hallucinated) - tp
tn = sum(1 for s in grounded if _verify_grounding(s, listing))
fp = len(grounded) - tn

recall = tp / (tp + fn) if (tp + fn) else 0
precision = tp / (tp + fp) if (tp + fp) else 0
print(f'Recall (hallucinations caught): {recall:.0%}  ({tp}/{tp+fn})')
print(f'Precision (catches are correct): {precision:.0%}  ({tp}/{tp+fp})')
print(f'False positives on grounded summaries: {fp}/{len(grounded)}')